# How many correlations can you actually estimate?

**Out of Sample** — [read the article](https://arthurnoo.pythonanywhere.com/how-many-correlations)

This notebook reproduces every figure and every number in the article.

## How to run it

**In your browser, with nothing to install:** [open it in Colab](https://colab.research.google.com/github/Arthurnoo/out-of-sample/blob/main/notebooks/covariance.ipynb),
then Runtime, Run all.

**On your own machine:** `git clone https://github.com/Arthurnoo/out-of-sample`,
open the folder in VS Code, choose a Python interpreter in the top right, and
Run All.

Either way the next cell installs whatever is missing. The first run downloads
prices and takes a minute or two.

## How to read it

Each section matches a section of the article. Run a cell, look at what it
prints, then read the note underneath explaining what to make of it.

**Section 0 holds every parameter you might want to change.** Everything after
it is written so that changing those values changes the whole notebook. That is
the point: the numbers below are mine, and the interesting question is what
happens to them when you move something.

Your numbers may differ slightly from the article if you run this later. The
data keeps growing, which is itself worth noticing.

## If you find a mistake

Tell me. There is a form at the bottom of the article, and a wrong number
corrected in public is worth more than a right one nobody checked.

In [ ]:
# Installs anything missing, then does nothing on later runs.
# This is what makes "Run All" work in Colab, in VS Code, or on a bare machine.
import importlib, subprocess, sys

for package in ['yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy']:
    try:
        importlib.import_module(package)
    except ImportError:
        print(f'installing {package} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

print('ready')

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

pd.set_option('display.width', 130)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.color': '#e4e9ee', 'axes.facecolor': 'white'})

## 0. The knobs

| Knob | What it does | Try |
| --- | --- | --- |
| `TICKERS` | the universe | cut it to 20 names and watch q fall |
| `START` | first day of data | `2010-01-01` for a longer sample |
| `WINDOWS` | estimation windows in the sweep | add `60` to push q above 1 |
| `HOLD` | how long a portfolio is held | `21` for monthly rebalancing |

The one that matters is the length of the estimation window, because it sets

```
q = number of assets / length of the window
```

and q decides everything in this piece. Below 0.1 the sample covariance is fine.
Above 0.5 it becomes actively harmful.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  PARAMETERS — the only cell you need to edit
# ─────────────────────────────────────────────────────────────────────
TICKERS = ['AAPL','MSFT','AMZN','GOOGL','META','NVDA','TSLA','JPM','BAC','WFC','GS','MS','C',
 'JNJ','PFE','MRK','ABBV','LLY','UNH','CVS','XOM','CVX','COP','SLB','PSX','PG','KO','PEP',
 'WMT','COST','HD','MCD','NKE','SBUX','TGT','LOW','DIS','CMCSA','VZ','T','TMUS','NFLX',
 'INTC','AMD','QCOM','TXN','AVGO','ORCL','CRM','ADBE','CSCO','IBM','HON','GE','CAT','DE',
 'BA','LMT','RTX','UPS','UNP','MMM','DOW','LIN','NEM','FCX','DUK','SO','NEE','AEP',
 'SPG','PLD','AMT','O','BLK','SCHW','AXP','V','MA','PYPL']

START   = '2016-01-01'
WINDOWS = [1260, 504, 252, 160, 120, 100, 90]
HOLD    = 63

## 1. Counting before estimating

With N assets there are N(N+1)/2 numbers in a covariance matrix. Compare that to
the N×T observations you have.

In [ ]:
rows = []
for N in [10, 50, 100, 500]:
    for T in [252, 1260]:
        p = N * (N + 1) // 2
        rows.append({'assets N': N, 'days T': T, 'parameters': p,
                     'observations': N * T, 'per parameter': N * T / p, 'q = N/T': N / T})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:14.2f}'))

Five hundred assets and one year of data give roughly **one observation per
parameter**. That is not a hard problem, it is a badly posed one.

## 2. The data

In [ ]:
frames = {}
for s in TICKERS:
    try:
        h = yf.Ticker(s).history(start=START, auto_adjust=True)
        if h.index.tz is not None:
            h.index = h.index.tz_localize(None)
        r = h['Close'].pct_change().dropna()
        if len(r) > 2000:
            frames[s] = r
    except Exception:
        pass

R = pd.DataFrame(frames).dropna()
N, T = R.shape[1], R.shape[0]
print(f"{N} stocks, {T} days, {R.index[0].date()} to {R.index[-1].date()}")
print(f"q = N/T = {N/T:.3f}    parameters to estimate: {N*(N+1)//2:,}")

## 3. Which correlations are real?

Suppose the returns had **no structure at all**. The eigenvalues of the sample
correlation matrix would still spread out, and Marchenko and Pastur (1967) gave
the exact shape of that spread. Everything falls between

```
λ± = (1 ± √q)²
```

Anything inside that band is indistinguishable from chance. Only what sticks out
above it carries information.

In [ ]:
def marchenko_pastur(q, n_points=600):
    lo, hi = (1 - np.sqrt(q))**2, (1 + np.sqrt(q))**2
    lam = np.linspace(max(lo, 1e-12), hi, n_points)
    dens = np.sqrt(np.clip((hi - lam) * (lam - lo), 0, None)) / (2 * np.pi * q * lam)
    return lam, dens, lo, hi

C = np.corrcoef(R.to_numpy().T)
vals, vecs = np.linalg.eigh(C)
order = np.argsort(vals)[::-1]
vals, vecs = vals[order], vecs[:, order]

q = N / T
lam, dens, lo, hi = marchenko_pastur(q)
inside = ((vals >= lo) & (vals <= hi)).sum()

print(f"noise band          : [{lo:.3f}, {hi:.3f}]")
print(f"eigenvalues inside  : {inside} of {N}  ({inside/N*100:.0f}%)")
print(f"largest eigenvalue  : {vals[0]:.2f}, which is {vals[0]/N*100:.1f}% of all variance")
print(f"next four           : {np.round(vals[1:5], 2)}")
same_sign = bool(np.all(np.sign(vecs[:,0]) == np.sign(vecs[0,0])))
print(f"\ntop eigenvector has every component with the same sign: {same_sign}")
print("That is the market itself: one factor that moves everything together.")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.hist(vals, bins=60, density=True, color='#8b98a6', alpha=.65,
        edgecolor='white', label='the real market')
ax.plot(lam, dens, color='#c83c50', lw=2, label='what pure noise would give')
ax.axvspan(lo, hi, color='#c83c50', alpha=.07)
ax.set_xlim(0, min(vals.max()*1.05, hi*3.2)); ax.legend()
ax.set_xlabel('eigenvalue'); ax.set_ylabel('density')
plt.show()

## 4. Where it breaks

The minimum-variance portfolio is the most demanding use of a covariance matrix,
because it requires **inverting** it, and inverting turns the smallest
eigenvalues into the largest numbers.

Shrinkage mixes the noisy sample matrix with a stable, structured target. The
mixing intensity is computed from the data, not chosen.

In [ ]:
def min_variance_weights(cov):
    inv = np.linalg.pinv(cov)
    w = inv @ np.ones(cov.shape[0])
    return w / w.sum()

def ledoit_wolf(X):
    """Shrink the sample covariance toward a constant-correlation target.
    delta is the optimal mixing intensity, computed from the data."""
    T_, N_ = X.shape
    Xc = X - X.mean(0)
    S = (Xc.T @ Xc) / T_
    var = np.diag(S); sd = np.sqrt(var)
    Rm = S / np.outer(sd, sd)
    r_bar = (Rm.sum() - N_) / (N_ * (N_ - 1))
    F = r_bar * np.outer(sd, sd); np.fill_diagonal(F, var)
    pi_mat = np.zeros((N_, N_))
    for t in range(T_):
        pi_mat += (np.outer(Xc[t], Xc[t]) - S) ** 2
    pi_mat /= T_
    gamma = ((F - S) ** 2).sum()
    delta = float(np.clip((pi_mat.sum() / gamma) / T_ if gamma > 0 else 0, 0, 1))
    return delta * F + (1 - delta) * S, delta

rows = []
for win in WINDOWS:
    if win >= len(R) - HOLD:
        continue
    out = {'window': win, 'q': N / win}
    for method in ['sample', 'shrunk', 'equal']:
        pnl, ds = [], []
        j = win
        while j + HOLD <= len(R):
            tr, te = R.iloc[j-win:j], R.iloc[j:j+HOLD]
            if method == 'equal':
                w = np.ones(N) / N
            elif method == 'sample':
                w = min_variance_weights(np.cov(tr.to_numpy().T))
            else:
                cov, d = ledoit_wolf(tr.to_numpy()); ds.append(d)
                w = min_variance_weights(cov)
            pnl.append(te.to_numpy() @ w); j += HOLD
        pnl = np.concatenate(pnl)
        out[f'vol {method} %'] = pnl.std(ddof=1) * np.sqrt(252) * 100
        if method == 'shrunk':
            out['shrinkage'] = np.mean(ds)
    rows.append(out)

sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False, float_format=lambda v: f'{v:14.3f}'))

fig, ax = plt.subplots(figsize=(7.4, 3.8))
ax.plot(sweep['q'], sweep['vol sample %'], marker='o', color='#c83c50', label='sample covariance')
ax.plot(sweep['q'], sweep['vol shrunk %'], marker='o', color='#0e8f5f', label='after shrinkage')
ax.plot(sweep['q'], sweep['vol equal %'], ls='--', color='#8b98a6', label='equal weights')
ax.set_xlabel('q = assets / window'); ax.set_ylabel('realised volatility (%)'); ax.legend()
plt.show()

Read the first column. The portfolio whose entire purpose is to minimise
volatility delivers the **worst** volatility once q approaches 1, far worse than
holding everything in equal weight and estimating nothing at all.

The optimiser is working perfectly. It is faithfully exploiting correlations
that are not there.

And look at the shrinkage column: nobody told it to, but as data per parameter
falls it decides by itself to trust the data less.

## 5. Things worth trying yourself

1. **Cut `TICKERS` to the first 20 names.** q falls by a factor of four. Does
   the sample covariance still blow up at a 90-day window?
2. **Add `60` and `45` to `WINDOWS`**, pushing q above 1. The matrix becomes
   singular and `pinv` quietly handles it. Look at what the weights do.
3. **Replace the shrinkage target** with the identity matrix instead of constant
   correlation. Does δ change? Which target wins?
4. **Add a no-short constraint** by clipping weights at zero and renormalising.
   Most of the damage disappears, which is exactly why constrained optimisers
   are popular and why this problem goes unnoticed.

Found something I got wrong? There is a form at the bottom of the article.